# 🎓 Vietnamese Text-to-SQL V5: Prompt-Aligned QLoRA Fine-Tuning

This notebook covers **fine-tuning only**: build prompt-aligned training data from MultiSpider-Vietnamese `train.json`, run QLoRA on Qwen2.5-Coder-7B-Instruct, and save the adapter to Google Drive. Inference, evaluation, and EM/ESM normalization live in a separate notebook.

| Setting | Value |
|---------|-------|
| Base model | `Qwen/Qwen2.5-Coder-7B-Instruct` |
| Adapter | QLoRA (rank=64, alpha=128, 7 target modules) |
| Embedding | `BAAI/bge-m3` (~2.3 GB VRAM) |
| Dataset | MultiSpider-Vietnamese (Multilingual Text-to-SQL benchmark) |

**V5: Prompt Alignment** — Training prompt matches inference (M-Schema, column linking, value injection, value hints, sample rows, English hint) with **20% feature dropout**.

**Adapted for Vietnamese** using the MultiSpider dataset (dreamerdeo/multispider).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

#@title 📦 Install Dependencies { display-mode: "form" }
!pip install -q transformers accelerate peft bitsandbytes trl datasets
!pip install -q sentencepiece sqlparse nltk sentence-transformers
!pip install -q gdown


## 📥 Download MultiSpider-Vietnamese Dataset

In [ ]:

#@title 📥 Download MultiSpider Dataset (Vietnamese) { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════════
# Downloads the MultiSpider dataset from HuggingFace using
# snapshot_download (NOT load_dataset, which fails due to
# inconsistent JSON schemas across per-database files).
#
# The HF repo structure is:
#   dataset/multispider/with_english_value/  → train/dev per language
#   dataset/spider/database/                 → SQLite DBs + tables.json
#   dataset/spider/database/*/examples_vi.json → per-DB Vietnamese Qs
# ═══════════════════════════════════════════════════════════════════

!pip install -q huggingface_hub

import os, json, glob, shutil, nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ All dependencies installed")

MULTISPIDER_DIR = "/content/multispider"
os.makedirs(MULTISPIDER_DIR, exist_ok=True)

# ── Step 1: Download the full HuggingFace repo ──────────────────
from huggingface_hub import snapshot_download

HF_LOCAL = "/content/hf_multispider"
if not os.path.exists(HF_LOCAL):
    print("📥 Downloading MultiSpider from HuggingFace (this may take a few minutes)...")
    snapshot_download(
        repo_id="dreamerdeo/multispider",
        repo_type="dataset",
        local_dir=HF_LOCAL,
    )
    print("✅ Download complete")
else:
    print("✅ MultiSpider repo already downloaded")

# ── Step 2: Locate Spider databases + tables.json ────────────────
# The repo has: dataset/spider/database/<db_name>/<db_name>.sqlite
SPIDER_DB_SRC = os.path.join(HF_LOCAL, "dataset", "spider", "database")
DB_DST = os.path.join(MULTISPIDER_DIR, "database")

if os.path.isdir(SPIDER_DB_SRC):
    if not os.path.isdir(DB_DST):
        # Symlink instead of copy to save disk space
        os.symlink(SPIDER_DB_SRC, DB_DST)
    print(f"✅ Spider databases linked: {DB_DST}")
    n_dbs = len([d for d in os.listdir(DB_DST) if os.path.isdir(os.path.join(DB_DST, d))])
    print(f"   {n_dbs} databases found")
else:
    print(f"⚠️ Spider database dir not found at {SPIDER_DB_SRC}")
    print(f"   Listing HF repo structure:")
    for root, dirs, files in os.walk(HF_LOCAL):
        depth = root.replace(HF_LOCAL, "").count(os.sep)
        if depth < 3:
            indent = " " * 2 * depth
            print(f"   {indent}{os.path.basename(root)}/")
            if depth == 2:
                for f in files[:5]:
                    print(f"   {indent}  {f}")

# Find tables.json — could be at dataset/spider/tables.json or dataset/tables.json
TABLES_JSON_PATH = os.path.join(MULTISPIDER_DIR, "tables.json")
if not os.path.exists(TABLES_JSON_PATH):
    candidates = glob.glob(os.path.join(HF_LOCAL, "**/tables.json"), recursive=True)
    # Prefer the top-level Spider tables.json (not per-database ones)
    # Per-database tables_*.json are small (~2KB); the main one is larger
    candidates.sort(key=lambda p: os.path.getsize(p), reverse=True)
    for c in candidates:
        # Skip per-language tables (tables_vi.json, tables_de.json etc.)
        basename = os.path.basename(c)
        if basename == "tables.json" and os.path.getsize(c) > 10000:
            shutil.copy2(c, TABLES_JSON_PATH)
            print(f"✅ Copied tables.json ({os.path.getsize(c)//1024}KB) from {c}")
            break
    if not os.path.exists(TABLES_JSON_PATH):
        # Fallback: use the xlangai/spider dataset tables.json
        print("⚠️ Main tables.json not found in HF repo, downloading from xlangai/spider...")
        from huggingface_hub import hf_hub_download
        hf_hub_download(
            repo_id="xlangai/spider",
            filename="spider/tables.json",
            repo_type="dataset",
            local_dir="/content/spider_fallback",
        )
        for c in glob.glob("/content/spider_fallback/**/tables.json", recursive=True):
            shutil.copy2(c, TABLES_JSON_PATH)
            print(f"✅ Copied tables.json from xlangai/spider")
            break

# ── Step 3: Find and load Vietnamese train/dev data ──────────────
# Strategy: look for Vietnamese JSON files in multiple possible locations
print("\n🔍 Searching for Vietnamese data files...")

vi_train = []
vi_dev = []

# --- Strategy A: Check dataset/multispider/with_english_value/ ---
# This is the recommended version per the MultiSpider paper
for subdir in ["with_english_value", "with_orginal_value"]:
    ms_dir = os.path.join(HF_LOCAL, "dataset", "multispider", subdir)
    if not os.path.isdir(ms_dir):
        continue
    print(f"   Found: dataset/multispider/{subdir}/")

    # Look for Vietnamese train/dev files
    vi_patterns = ["*vi*train*", "*train*vi*", "*vi*.json"]
    for pattern in vi_patterns:
        for fp in glob.glob(os.path.join(ms_dir, "**", pattern), recursive=True):
            print(f"     📄 {os.path.relpath(fp, HF_LOCAL)}")

    # Load any JSON files containing Vietnamese data
    for fp in sorted(glob.glob(os.path.join(ms_dir, "**", "*.json"), recursive=True)):
        fname = os.path.basename(fp).lower()
        if "vi" not in fname:
            continue
        try:
            with open(fp, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                print(f"     ✅ Loaded {fname}: {len(data)} samples")
                # Determine train or dev
                if "train" in fname:
                    vi_train.extend(data)
                elif "dev" in fname or "test" in fname or "val" in fname:
                    vi_dev.extend(data)
                else:
                    # Check parent directory name
                    parent = os.path.basename(os.path.dirname(fp)).lower()
                    if "train" in parent:
                        vi_train.extend(data)
                    else:
                        vi_dev.extend(data)
        except Exception as e:
            print(f"     ⚠️ Failed to load {fname}: {e}")

    if vi_train or vi_dev:
        print(f"   ✅ Found data in {subdir}/")
        break  # Use with_english_value if available

# --- Strategy B: If no centralized files, collect per-database examples_vi.json ---
if not vi_train and not vi_dev:
    print("   ℹ️ No centralized Vietnamese files found, collecting per-database examples_vi.json...")

    # Per-database files: dataset/spider/database/<db_name>/examples_vi.json
    vi_all_examples = []
    for vi_file in sorted(glob.glob(os.path.join(HF_LOCAL, "**", "examples_vi.json"), recursive=True)):
        try:
            with open(vi_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                vi_all_examples.extend(data)
        except Exception:
            continue

    if vi_all_examples:
        print(f"   ✅ Collected {len(vi_all_examples)} Vietnamese examples from per-database files")
        # Split into train/dev using the same db_ids as original Spider
        # Spider dev DBs are a specific subset; use a simple 87/13 split matching Spider proportions
        # Better: check if there's an existing train/test split marker
        dev_dbs = set()
        # Try to identify dev databases from English dev files
        for dev_candidate in glob.glob(os.path.join(HF_LOCAL, "**", "dev*.json"), recursive=True):
            try:
                with open(dev_candidate, "r", encoding="utf-8") as f:
                    dev_data = json.load(f)
                if isinstance(dev_data, list) and len(dev_data) > 100:
                    dev_dbs = set(d.get("db_id", "") for d in dev_data)
                    if dev_dbs:
                        print(f"   ✅ Found dev split DB IDs from {os.path.basename(dev_candidate)}: {len(dev_dbs)} DBs")
                        break
            except Exception:
                continue

        if dev_dbs:
            for ex in vi_all_examples:
                if ex.get("db_id", "") in dev_dbs:
                    vi_dev.append(ex)
                else:
                    vi_train.append(ex)
        else:
            # Fallback: use all as training, will need manual dev split
            vi_train = vi_all_examples
            print("   ⚠️ Could not determine train/dev split — all samples in train")

# ── Step 4: Normalize field names and save ────────────────────────
def normalize_sample(sample):
    """Normalize field names to match pipeline expectations."""
    return {
        "db_id": sample.get("db_id", ""),
        "Vietnamese": sample.get("question", sample.get("Vietnamese", "")),
        "question": sample.get("question_en", sample.get("question_toks", "")),
        "query": sample.get("query", ""),
    }

# Check what fields exist in the data
if vi_train:
    sample_fields = list(vi_train[0].keys())
    print(f"\n   Sample fields: {sample_fields}")
    print(f"   Example: {json.dumps(vi_train[0], ensure_ascii=False)[:200]}")

    # If 'question' field contains Vietnamese (not English), adjust
    if vi_train[0].get("question", "") and not vi_train[0].get("Vietnamese", ""):
        # Standard MultiSpider format: 'question' has the Vietnamese text
        for lst in [vi_train, vi_dev]:
            for s in lst:
                s["Vietnamese"] = s.get("question", "")
                # English might be in a separate field or missing
                s["question"] = s.get("question_en", "")

# Save as JSON for compatibility with rest of pipeline
with open(os.path.join(MULTISPIDER_DIR, "train.json"), "w", encoding="utf-8") as f:
    json.dump(vi_train, f, ensure_ascii=False, indent=2)
with open(os.path.join(MULTISPIDER_DIR, "dev.json"), "w", encoding="utf-8") as f:
    json.dump(vi_dev, f, ensure_ascii=False, indent=2)

print(f"\n   ✅ Vietnamese train: {len(vi_train)} samples")
print(f"   ✅ Vietnamese dev:   {len(vi_dev)} samples")

# ── Step 5: Verify all required files ─────────────────────────────
for f_name in ["train.json", "dev.json", "tables.json"]:
    path = os.path.join(MULTISPIDER_DIR, f_name)
    assert os.path.exists(path), f"❌ Missing: {path}"
    print(f"   ✅ {f_name}")

# Verify databases
if os.path.isdir(DB_DST):
    n_sqlite = len(glob.glob(os.path.join(DB_DST, "**", "*.sqlite"), recursive=True))
    print(f"   ✅ {n_sqlite} SQLite database files")
else:
    print(f"   ⚠️ Database directory not found — execution-based evaluation will be limited")

# Show sample data
if vi_train:
    s = vi_train[0]
    print(f"\n   📋 Sample train entry:")
    print(f"      db_id:      {s.get('db_id', 'N/A')}")
    print(f"      Vietnamese: {str(s.get('Vietnamese', ''))[:80]}")
    print(f"      English:    {str(s.get('question', ''))[:80]}")
    print(f"      SQL:        {str(s.get('query', ''))[:80]}")

print(f"\n✅ MultiSpider Vietnamese dataset ready at {MULTISPIDER_DIR}")


## 📦 Core Imports & Utilities

In [ ]:

import json, re, os, math, gc, sqlite3, time
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import Counter, defaultdict
import sqlparse
from tqdm.auto import tqdm

DATA_DIR = Path("./data"); DATA_DIR.mkdir(exist_ok=True)


## 🗄️ Schema Loading & SQL Execution

In [ ]:

import glob

def spider_tables_json_to_ddl(tables_json_data: List[Dict]) -> Dict[str, str]:
    """Convert Spider-format tables.json into {db_id: CREATE TABLE DDL}."""
    db_schemas = {}
    for db in tables_json_data:
        db_id = db["db_id"]
        table_names = db["table_names_original"]
        col_names = db["column_names_original"]
        col_types = db.get("column_types", [])
        pks = set(db.get("primary_keys", []))
        fks = db.get("foreign_keys", [])

        table_cols = defaultdict(list)
        for col_idx, (tbl_idx, col_name) in enumerate(col_names):
            if tbl_idx == -1:
                continue
            ctype = col_types[col_idx] if col_idx < len(col_types) else "TEXT"
            sql_type = {
                "text": "TEXT", "number": "REAL", "time": "TEXT",
                "boolean": "INTEGER", "others": "TEXT",
            }.get(ctype.lower(), "TEXT")
            is_pk = col_idx in pks
            table_cols[tbl_idx].append((col_name, sql_type, is_pk))

        fk_map = {}
        for fk_col, ref_col in fks:
            if ref_col < len(col_names):
                ref_tbl_idx, ref_col_name = col_names[ref_col]
                if ref_tbl_idx >= 0:
                    fk_map[fk_col] = (table_names[ref_tbl_idx], ref_col_name)

        ddl_parts = []
        for tbl_idx, tbl_name in enumerate(table_names):
            cols_sql = []
            for col_name, sql_type, is_pk in table_cols.get(tbl_idx, []):
                line = f"    {col_name} {sql_type}"
                if is_pk:
                    line += " PRIMARY KEY"
                cols_sql.append(line)
            fk_lines = []
            for col_idx, (tbl_i, cn) in enumerate(col_names):
                if tbl_i == tbl_idx and col_idx in fk_map:
                    ref_tbl, ref_col = fk_map[col_idx]
                    fk_lines.append(
                        f"    FOREIGN KEY ({cn}) REFERENCES {ref_tbl}({ref_col})"
                    )
            all_lines = cols_sql + fk_lines
            if not all_lines:
                all_lines = ["    id INTEGER PRIMARY KEY"]
            ddl = f"CREATE TABLE {tbl_name} (\n" + ",\n".join(all_lines) + "\n);"
            ddl_parts.append(ddl)

        db_schemas[db_id] = "\n\n".join(ddl_parts)
    return db_schemas


def execute_sql_on_db(db_path: str, sql: str, timeout=30) -> Tuple[bool, Any]:
    """Execute against a real SQLite database file (read-only)."""
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=timeout)
        conn.execute("PRAGMA busy_timeout = 5000")
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description] if cur.description else []
        conn.close()
        return True, {"columns": cols, "rows": rows}
    except Exception as e:
        return False, str(e)


def compare_results(r1, r2) -> bool:
    """Compare execution results as unordered sets."""
    if r1 is None or r2 is None:
        return False
    try:
        rows1 = r1["rows"] if isinstance(r1, dict) else r1
        rows2 = r2["rows"] if isinstance(r2, dict) else r2
        set1 = set(tuple(sorted(str(v) for v in r)) for r in rows1)
        set2 = set(tuple(sorted(str(v) for v in r)) for r in rows2)
        if set1 == set2:
            return True
        if len(rows1) == len(rows2):
            norm1 = sorted(tuple(sorted(str(v) for v in r)) for r in rows1)
            norm2 = sorted(tuple(sorted(str(v) for v in r)) for r in rows2)
            if norm1 == norm2:
                return True
            def _nv(v):
                s = str(v).strip()
                try:
                    f = float(s)
                    return str(int(f)) if f == int(f) else f"{f:.6f}"
                except (ValueError, OverflowError):
                    return s.lower()
            n1 = sorted(tuple(sorted(_nv(v) for v in r)) for r in rows1)
            n2 = sorted(tuple(sorted(_nv(v) for v in r)) for r in rows2)
            return n1 == n2
        return False
    except Exception:
        return str(r1) == str(r2)


# Load schemas
with open(os.path.join(MULTISPIDER_DIR, "tables.json"), "r", encoding="utf-8") as f:
    _tables_json = json.load(f)
multispider_vi_schemas = spider_tables_json_to_ddl(_tables_json)
print(f"✅ Loaded schemas: {len(multispider_vi_schemas)} databases")

# Find database files
db_paths = {}
db_base = ""
for candidate in ["databases", "database"]:
    p = os.path.join(MULTISPIDER_DIR, candidate)
    if os.path.isdir(p):
        db_base = p
        break
if db_base:
    for db_name in os.listdir(db_base):
        db_dir = os.path.join(db_base, db_name)
        if os.path.isdir(db_dir):
            for ext in ["*.sqlite", "*.db", "*.sqlite3"]:
                found = glob.glob(os.path.join(db_dir, ext))
                if found:
                    db_paths[db_name] = found[0]
                    break
print(f"✅ Found {len(db_paths)} SQLite database files")


## ⚙️ Configuration

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

#@title ⚙️ Configuration { display-mode: "form" }

FINETUNE_7B_BASE  = "Qwen/Qwen2.5-Coder-7B-Instruct"
LORA_RANK_7B      = 64        #@param {type:"integer"}
LORA_ALPHA_7B     = 128        #@param {type:"integer"}
NUM_EPOCHS_7B     = 3         #@param {type:"slider", min:1, max:10, step:1}
LEARNING_RATE_7B  = 5e-5      #@param {type:"number"}
PER_DEVICE_BATCH_7B = 6      #@param {type:"integer"}
GRAD_ACCUM_7B     = 4        #@param {type:"integer"}  # raised 4→16 to compensate for smaller batch
MAX_SEQ_LENGTH_7B = 4096      #@param {type:"integer"}
USE_ENHANCED_PROMPT = True     #@param {type:"boolean"}

ADAPTER_7B_OUTPUT = Path("./data/vietnamese_7b_adapter")
ADAPTER_7B_OUTPUT.mkdir(parents=True, exist_ok=True)

# ── Prompt template (SQL-only, no CoT) ────────────────────────
SQL_ONLY_PROMPT = '''Database Engine: SQLite

Database Schema:
{db_details}

Question:
{question}

Generate the SQL query that answers this question. Output ONLY the SQL inside a code block:
```sql
-- Your SQL query
```'''

SQL_ONLY_SYSTEM = (
    "You are an expert Vietnamese-to-SQL translator. Given a database schema "
    "and a question (in Vietnamese or English), carefully map Vietnamese terms to "
    "the correct English column and table names in the schema, then output "
    "ONLY the correct SQL query inside a code block. No explanations."
)

# ── Enhanced prompt template (with table summary + JOIN guidance) ─
SQL_ENHANCED_PROMPT = '''Database Engine: SQLite

Database Schema:
{db_details}

Table → Column Reference (use ONLY columns from the table that has them):
{table_summary}

IMPORTANT: Use the MINIMUM number of tables needed.
If ALL required columns exist in ONE table, do NOT use JOIN.

Question:
{question}

Generate the SQL query that answers this question. Output ONLY the SQL inside a code block:
```sql
-- Your SQL query
```'''


def build_table_column_summary(schema_ddl: str) -> str:
    """Generate a compact table→columns mapping for the enhanced prompt."""
    tables = {}
    cur_table = None
    for line in schema_ddl.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    if not tables:
        return ""
    lines = []
    for table, cols in tables.items():
        lines.append(f"  • {table}: {', '.join(cols)}")
    return "\n".join(lines)


def format_prompt(schema: str, question: str) -> str:
    """Format prompt using the configured template (base or enhanced)."""
    if USE_ENHANCED_PROMPT:
        table_summary = build_table_column_summary(schema)
        return SQL_ENHANCED_PROMPT.format(
            db_details=schema[:4500],
            table_summary=table_summary,
            question=question,
        )
    else:
        return SQL_ONLY_PROMPT.format(
            db_details=schema[:5000],
            question=question,
        )

print(f"✅ Config ready")
print(f"   Base: {FINETUNE_7B_BASE}")
print(f"   LoRA: rank={LORA_RANK_7B}, alpha={LORA_ALPHA_7B}")
print(f"   LR: {LEARNING_RATE_7B}, Epochs: {NUM_EPOCHS_7B}")
print(f"   Effective batch: {PER_DEVICE_BATCH_7B * GRAD_ACCUM_7B}")
print(f"   Prompt: {'🆕 Enhanced (table summary + JOIN guidance)' if USE_ENHANCED_PROMPT else '📋 Base (schema only)'}")


---
## 🚀 Section A: Tier 1 Feature Configuration (Training)

Moves all technique toggles before training so the prompt builder can use them.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 📋 DOCUMENTED FULL PIPELINE (Option B — headline configuration)
# ═══════════════════════════════════════════════════════════════════
# All original components ACTIVE, exactly as in the SOTA runs:
#   5 prompt components + multi-temp voting (shortest-SQL winner)
#   + self-correction + ID fixing + filter removal + value grounding
#   + low-confidence 8-extra-candidate fallback.
# The revised manuscript documents ALL of these (Sections 3.1–3.8).
# Additions in this notebook are logging/reproducibility only and do
# not change pipeline behavior:
#   per-candidate logs, deterministic seeds, provenance metadata.
# NOTE: original SOTA runs were unseeded; this re-run will land within
# sampling noise of the published numbers and BECOMES the reported run.
# ═══════════════════════════════════════════════════════════════════


# Paste as a NEW CELL after your existing ⚙️ Configuration cell.
# This moves the technique flags before training so the prompt
# builder can use them during training data construction.
# ═══════════════════════════════════════════════════════════════════



#@title 🚀 Tier 1 Feature Configuration (Shared: Train + Inference) { display-mode: "form" }

# ── Prompt Alignment ────────────────────────────────────────────
TRAIN_USE_ALIGNED_PROMPT  = True    #@param {type:"boolean"}
FEATURE_DROPOUT_RATE      = 0.20   #@param {type:"number"}

# ── Technique Toggles (used by BOTH training prompt and inference) ─
ENABLE_MSCHEMA            = True   #@param {type:"boolean"}
ENABLE_VALUE_HINTS        = True   #@param {type:"boolean"}
VALUE_HINT_MAX_PER_COL    = 4      #@param {type:"integer"}

ENABLE_COLUMN_LINKING     = True   #@param {type:"boolean"}
COL_LINK_TOP_K            = 6      #@param {type:"integer"}

ENABLE_VI_COL_DESC        = True   #@param {type:"boolean"}
VI_COL_DESC_PATH          = "/content/drive/MyDrive/vi_column_descriptions.json"  #@param {type:"string"}

ENABLE_ENGLISH_HINT       = True   #@param {type:"boolean"}

ENABLE_BILINGUAL_SCHEMA   = True   #@param {type:"boolean"}
VI_TABLE_GLOSS_PATH       = "/content/drive/MyDrive/vi_table_glosses.json"  #@param {type:"string"}

ENABLE_VALUE_INJECTION    = True   #@param {type:"boolean"}
VALUE_INJECTION_TOP_K     = 5      #@param {type:"integer"}
VALUE_INJECTION_THRESHOLD = 0.40   #@param {type:"number"}

ENABLE_BILINGUAL_EMBEDDINGS = True   #@param {type:"boolean"}

ENABLE_SAMPLE_ROWS        = True   #@param {type:"boolean"}
SAMPLE_ROWS_LIMIT         = 3      #@param {type:"integer"}
SAMPLE_ROWS_MAX_TABLES    = 3      #@param {type:"integer"}

print("🚀 Tier 1 Feature Configuration:")
print(f"   Prompt alignment for training : {'✅ ON' if TRAIN_USE_ALIGNED_PROMPT else '❌ OFF'}")
print(f"   Feature dropout rate          : {FEATURE_DROPOUT_RATE}")
print(f"   M-Schema                      : {'✅ ON' if ENABLE_MSCHEMA else '❌ OFF'}")
print(f"   Value Hints                   : {'✅ ON' if ENABLE_VALUE_HINTS else '❌ OFF'}")
print(f"   Column Linking (embedding)    : {'✅ ON' if ENABLE_COLUMN_LINKING else '❌ OFF'}")
print(f"   Vietnamese Column Descriptions    : {'✅ ON' if ENABLE_VI_COL_DESC else '❌ OFF'}")
print(f"   English Translation Hint      : {'✅ ON' if ENABLE_ENGLISH_HINT else '❌ OFF'}")
print(f"   Bilingual Schema (AR gloss)   : {'✅ ON' if ENABLE_BILINGUAL_SCHEMA else '❌ OFF'}")
print(f"   Value Injection (emb match)   : {'✅ ON' if ENABLE_VALUE_INJECTION else '❌ OFF'}")
print(f"   Bilingual Embeddings          : {'✅ ON' if ENABLE_BILINGUAL_EMBEDDINGS else '❌ OFF'}")
print(f"   Sample Rows in Prompt         : {'✅ ON' if ENABLE_SAMPLE_ROWS else '❌ OFF'}")


In [ ]:
#@title ✅ Training Configuration Sanity Check { display-mode: "form" }
# Asserts all prompt components are ON (headline training configuration).
_off = [f for f in ["TRAIN_USE_ALIGNED_PROMPT","ENABLE_MSCHEMA","ENABLE_VALUE_HINTS",
                    "ENABLE_COLUMN_LINKING","ENABLE_AR_COL_DESC","ENABLE_ENGLISH_HINT",
                    "ENABLE_BILINGUAL_SCHEMA","ENABLE_VALUE_INJECTION","ENABLE_SAMPLE_ROWS"]
        if not globals().get(f, False)]
if _off:
    raise RuntimeError("TRAINING CONFIG VIOLATION — these must be True: " + ", ".join(_off))
print("✅ Training configuration verified — all prompt components ACTIVE:")
print(f"   Feature dropout rate: {FEATURE_DROPOUT_RATE}")


---
## 🧰 Section B: Shared Function Definitions (Train + Inference)

These functions are used by training prompt construction: schema parsing, M-Schema, column linking, value hints, value injection, sample rows, prompt templates.

In [ ]:

#@title 🧰 Shared: Schema Parsing, M-Schema, Lookups { display-mode: "form" }

# Paste as a NEW CELL after Section A.
# These functions are used by BOTH training prompt construction
# and inference. They are identical to the originals in your
# notebook — just defined earlier so training can use them.
# ═══════════════════════════════════════════════════════════════════



#@title 🧰 Shared Functions (Train + Inference) { display-mode: "form" }

import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field

# ── EvalSample dataclass (reused for both train and eval) ───────
@dataclass
class EvalSample:
    id: str
    db_id: str
    vietnamese_question: str
    gold_sql: str
    schema_ddl: str
    db_path: str = ""
    english_question: str = ""


# ── Schema parsing ──────────────────────────────────────────────

def extract_tables_columns(schema_sql: str) -> Dict[str, List[str]]:
    """Parse CREATE TABLE DDL into {table_name: [column_names]}."""
    tables = {}
    cur_table = None
    for line in schema_sql.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I,
        )
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+\w+', line)
            if cm and not any(
                line.upper().startswith(kw) for kw in
                ('PRIMARY', 'FOREIGN', 'UNIQUE', 'CHECK', 'CONSTRAINT', '--')
            ):
                tables[cur_table].append(cm.group(1))
    return tables


def build_table_column_summary(schema_ddl: str) -> str:
    """Compact table→columns mapping for prompt."""
    tables = extract_tables_columns(schema_ddl)
    if not tables:
        return ""
    lines = []
    for table, cols in tables.items():
        lines.append(f"  • {table}: {', '.join(cols)}")
    return "\n".join(lines)


# ── Index tables.json by db_id ──────────────────────────────────
_tables_json_by_id = {}
try:
    for _entry in _tables_json:
        _tables_json_by_id[_entry["db_id"]] = _entry
    print(f"✅ Indexed {len(_tables_json_by_id)} database schemas for M-Schema")
except NameError:
    print("⚠️ _tables_json not found — will fall back to DDL-based M-Schema")


# ── Vietnamese description + gloss lookups (populated in Section D) ─
_vi_col_descriptions = {}  # {db_id: {"table.column": "mô tả tiếng Việt", ...}}
_vi_table_glosses    = {}  # {db_id: {"table_name": "mô tả tiếng Việt", ...}}


def get_column_description(db_id: str, table: str, column: str) -> str:
    """Look up Vietnamese description for a column. Returns '' if not found."""
    return _vi_col_descriptions.get(db_id, {}).get(f"{table}.{column}", "")


def get_table_gloss(db_id: str, table_name: str) -> str:
    """Look up Vietnamese gloss for a table. Returns '' if not found."""
    return _vi_table_glosses.get(db_id, {}).get(table_name, "")


# ── M-Schema builder ───────────────────────────────────────────

def build_mschema(tables_json_entry: dict, db_id: str = "") -> str:
    """Build M-Schema from a single tables.json database entry.
    Includes Vietnamese table glosses and column descriptions inline."""
    table_names = tables_json_entry["table_names_original"]
    col_names   = tables_json_entry["column_names_original"]
    col_types   = tables_json_entry.get("column_types", [])
    pks         = set(tables_json_entry.get("primary_keys", []))
    fks         = tables_json_entry.get("foreign_keys", [])

    table_cols = defaultdict(list)
    for col_idx, (tbl_idx, col_name) in enumerate(col_names):
        if tbl_idx == -1:
            continue
        ctype = col_types[col_idx] if col_idx < len(col_types) else "text"
        sql_type = {"text": "TEXT", "number": "REAL", "time": "TEXT",
                    "boolean": "INT", "others": "TEXT"}.get(ctype.lower(), "TEXT")
        is_pk = col_idx in pks
        table_cols[tbl_idx].append((col_name, sql_type, is_pk, col_idx))

    fk_map = {}
    for fk_col, ref_col in fks:
        if ref_col < len(col_names):
            ref_tbl_idx, ref_col_name = col_names[ref_col]
            if ref_tbl_idx >= 0:
                fk_map[fk_col] = (table_names[ref_tbl_idx], ref_col_name)

    lines = []
    for tbl_idx, tbl_name in enumerate(table_names):
        cols = table_cols.get(tbl_idx, [])
        if not cols:
            continue
        col_parts = []
        for col_name, sql_type, is_pk, col_idx in cols:
            pk_mark = "*" if is_pk else ""
            ar_desc = ""
            if ENABLE_BILINGUAL_SCHEMA and db_id:
                ar_desc = get_column_description(db_id, tbl_name, col_name)
            if ar_desc:
                col_parts.append(f"({col_name}{pk_mark}, {sql_type}, {ar_desc})")
            else:
                col_parts.append(f"({col_name}{pk_mark}, {sql_type})")
        tbl_gloss = ""
        if ENABLE_BILINGUAL_SCHEMA and db_id:
            tbl_gloss = get_table_gloss(db_id, tbl_name)
        if tbl_gloss:
            lines.append(f"【{tbl_name}】 ({tbl_gloss})")
        else:
            lines.append(f"【{tbl_name}】")
        lines.append("  " + "  ".join(col_parts))
        for col_name, sql_type, is_pk, col_idx in cols:
            if col_idx in fk_map:
                ref_tbl, ref_col = fk_map[col_idx]
                lines.append(f"  -> FK: {col_name} -> {ref_tbl}.{ref_col}")
    return "\n".join(lines)


def build_mschema_from_ddl(schema_ddl: str) -> str:
    """Fallback: build M-Schema from DDL string (no bilingual annotations)."""
    tables = {}
    cur_table = None
    fks_list = []
    for line in schema_ddl.split("\n"):
        line = line.strip()
        m = re.match(
            r'CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?[`"\[]?([\w]+)[`"\]]?',
            line, re.I)
        if m:
            cur_table = m.group(1)
            tables[cur_table] = []
        elif cur_table and line and not line.startswith(')'):
            fk_m = re.match(
                r'FOREIGN\s+KEY\s*\((\w+)\)\s*REFERENCES\s+(\w+)\s*\((\w+)\)',
                line, re.I)
            if fk_m:
                fks_list.append((cur_table, fk_m.group(1), fk_m.group(2), fk_m.group(3)))
                continue
            if any(line.upper().startswith(kw) for kw in
                   ('PRIMARY','FOREIGN','UNIQUE','CHECK','CONSTRAINT','--')):
                continue
            cm = re.match(r'[`"\[]?([\w]+)[`"\]]?\s+(\w+)', line)
            if cm:
                col_name = cm.group(1)
                col_type = cm.group(2).upper()
                is_pk = 'PRIMARY KEY' in line.upper()
                tables[cur_table].append((col_name, col_type, is_pk))

    lines = []
    for tbl_name, cols in tables.items():
        col_parts = []
        for col_name, col_type, is_pk in cols:
            pk_mark = "*" if is_pk else ""
            col_parts.append(f"({col_name}{pk_mark}, {col_type})")
        lines.append(f"【{tbl_name}】")
        lines.append("  " + "  ".join(col_parts))
        for fk_tbl, fk_col, ref_tbl, ref_col in fks_list:
            if fk_tbl == tbl_name:
                lines.append(f"  -> FK: {fk_col} -> {ref_tbl}.{ref_col}")
    return "\n".join(lines)


def get_mschema(db_id: str, schema_ddl: str) -> str:
    """Get M-Schema for a database (from tables.json if available, else DDL)."""
    if db_id in _tables_json_by_id:
        return build_mschema(_tables_json_by_id[db_id], db_id=db_id)
    return build_mschema_from_ddl(schema_ddl)

#@title 🧰 Shared: Value Hints, Column Linking, Value Injection, Sample Rows { display-mode: "form" }

# ── Database value hints (question-aware) ───────────────────────

def sample_db_values_question_aware(db_path: str, schema_ddl: str,
                                     question: str,
                                     max_per_col: int = 4) -> str:
    """Question-aware value sampling from real DB. Returns compact hint string."""
    if not db_path or not os.path.exists(db_path):
        return ""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return ""

    all_hints = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl_name, columns in schema_tables.items():
            for col_name in columns:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl_name}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col_name.lower(), "TEXT").upper()
                    is_text = any(t in col_type for t in
                                  ["TEXT", "VARCHAR", "CHAR", "CLOB"])
                    if not is_text:
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col_name}" FROM "{tbl_name}" '
                        f'WHERE "{col_name}" IS NOT NULL AND TRIM("{col_name}") != "" '
                        f'LIMIT {max_per_col * 2}')
                    values = [str(row[0]) for row in cursor.fetchall()
                              if row[0] is not None and str(row[0]).strip()]
                    if not values:
                        continue
                    col_lower = col_name.lower()
                    relevance = 0
                    high_value = ['name', 'type', 'status', 'country', 'city',
                                  'state', 'region', 'continent', 'language',
                                  'nationality', 'category', 'genre', 'major',
                                  'department', 'sex', 'gender', 'color',
                                  'brand', 'title', 'position', 'head']
                    if any(hv in col_lower for hv in high_value):
                        relevance += 2
                    for v in values:
                        if v.lower() in question.lower():
                            relevance += 5
                    display = [v[:35] for v in values[:max_per_col]]
                    all_hints.append((relevance, f"    {tbl_name}.{col_name}: {display}"))
                except Exception:
                    continue
        conn.close()
    except Exception:
        return ""

    if not all_hints:
        return ""
    all_hints.sort(key=lambda x: -x[0])
    selected = [h[1] for h in all_hints[:min(len(all_hints), 15)]]
    return "Sample column values (use these EXACT values in your SQL):\n" + "\n".join(selected)


# ── Column linking (embedding similarity) ──────────────────────
# Requires _emb_model — loaded in Section D.  Functions degrade
# gracefully to no-ops when _emb_model is None.

_emb_model    = None   # set in Section D
_col_emb_cache = {}    # {db_id: {"candidates": [...], "embeddings": np.ndarray}}


def _build_column_description(table_name: str, col_name: str,
                               db_id: str = "") -> str:
    """Build natural-language description for embedding.
    Includes Vietnamese description for bilingual matching."""
    readable = re.sub(r'([a-z])([A-Z])', r'\1 \2', col_name)
    readable = readable.replace('_', ' ').lower()
    tbl_readable = re.sub(r'([a-z])([A-Z])', r'\1 \2', table_name)
    tbl_readable = tbl_readable.replace('_', ' ').lower()
    base = f"{readable} in {tbl_readable}"
    if ENABLE_BILINGUAL_EMBEDDINGS and db_id:
        ar_desc = get_column_description(db_id, table_name, col_name)
        if ar_desc:
            return f"{base} — {ar_desc}"
    return base


def _get_column_embeddings(db_id: str, schema_ddl: str) -> dict:
    """Get or compute cached column embeddings for a database."""
    if db_id in _col_emb_cache:
        return _col_emb_cache[db_id]

    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables or _emb_model is None:
        empty = {"candidates": [], "embeddings": np.array([])}
        _col_emb_cache[db_id] = empty
        return empty

    candidates = []
    texts = []
    for tbl, cols in schema_tables.items():
        for col in cols:
            desc = _build_column_description(tbl, col, db_id=db_id)
            candidates.append({
                "ref": f"{tbl}.{col}", "text": desc,
                "tbl": tbl, "col": col,
            })
            texts.append(desc)

    if texts:
        embeddings = _emb_model.encode(
            texts, normalize_embeddings=True,
            show_progress_bar=False, batch_size=64,
        )
    else:
        embeddings = np.array([])

    result = {"candidates": candidates, "embeddings": embeddings}
    _col_emb_cache[db_id] = result
    return result


def link_columns_to_question(db_id: str, schema_ddl: str, question: str,
                              db_path: str = "", top_k: int = 6) -> str:
    """Score every column against the Vietnamese question using embedding similarity.
    Returns a prompt hint with the top-K most relevant columns."""
    if _emb_model is None:
        return ""

    col_data = _get_column_embeddings(db_id, schema_ddl)
    candidates = col_data["candidates"]
    col_embeddings = col_data["embeddings"]

    if not candidates or len(col_embeddings) == 0:
        return ""

    q_embedding = _emb_model.encode(
        [question], normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    final_scores = col_embeddings @ q_embedding

    ranked_indices = np.argsort(-final_scores)[:top_k]
    lines = []
    for idx in ranked_indices:
        if final_scores[idx] < 0.50:  # raised to 0.50 — only strong matches survive
            continue
        c = candidates[idx]
        ar_desc = ""
        if ENABLE_VI_COL_DESC:
            ar_desc = get_column_description(db_id, c["tbl"], c["col"])
        if ar_desc:
            lines.append(f"  ★ {c['ref']} — {ar_desc}")
        else:
            lines.append(f"  ★ {c['ref']}")

    if not lines:
        return ""
    return "Likely relevant columns (prefer these):\n" + "\n".join(lines)


# ── Question-matched value injection (embedding-based) ─────────

_db_value_emb_cache = {}  # {db_id: {"values": [...], "embeddings": np.ndarray, "col_refs": [...]}}


def _build_value_embeddings(db_id: str, db_path: str, schema_ddl: str) -> dict:
    """Pre-compute embeddings for all text values in a database."""
    if db_id in _db_value_emb_cache:
        return _db_value_emb_cache[db_id]

    empty = {"values": [], "embeddings": np.array([]), "col_refs": []}
    if not db_path or not os.path.exists(db_path) or _emb_model is None:
        _db_value_emb_cache[db_id] = empty
        return empty

    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        _db_value_emb_cache[db_id] = empty
        return empty

    all_values = []
    col_refs = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl, cols in schema_tables.items():
            for col in cols:
                try:
                    cursor.execute(f'PRAGMA table_info("{tbl}")')
                    col_info = {row[1].lower(): row[2] for row in cursor.fetchall()}
                    col_type = col_info.get(col.lower(), "TEXT").upper()
                    if not any(t in col_type for t in ["TEXT", "VARCHAR", "CHAR", "CLOB"]):
                        continue
                    cursor.execute(
                        f'SELECT DISTINCT "{col}" FROM "{tbl}" '
                        f'WHERE "{col}" IS NOT NULL AND TRIM("{col}") != "" '
                        f'LIMIT 150')
                    for row in cursor.fetchall():
                        val = str(row[0]).strip()
                        if len(val) >= 2:
                            all_values.append(val)
                            col_refs.append(f"{tbl}.{col}")
                except Exception:
                    continue
        conn.close()
    except Exception:
        _db_value_emb_cache[db_id] = empty
        return empty

    if not all_values:
        _db_value_emb_cache[db_id] = empty
        return empty

    # Deduplicate
    seen = {}
    unique_values = []
    unique_refs = []
    for val, ref in zip(all_values, col_refs):
        key = val.lower()
        if key not in seen:
            seen[key] = len(unique_values)
            unique_values.append(val)
            unique_refs.append(ref)

    embeddings = _emb_model.encode(
        unique_values, normalize_embeddings=True,
        show_progress_bar=False, batch_size=128,
    )

    result = {"values": unique_values, "embeddings": embeddings, "col_refs": unique_refs}
    _db_value_emb_cache[db_id] = result
    return result


def find_question_matched_values(db_id: str, db_path: str, schema_ddl: str,
                                  vietnamese_question: str, english_question: str = "",
                                  top_k: int = 5, threshold: float = 0.40) -> str:
    """Find DB values semantically closest to the question.
    Uses BOTH Vietnamese and English queries for dual-language matching."""
    if _emb_model is None:
        return ""

    val_data = _build_value_embeddings(db_id, db_path, schema_ddl)
    values = val_data["values"]
    val_embeddings = val_data["embeddings"]
    col_refs = val_data["col_refs"]

    if not values or len(val_embeddings) == 0:
        return ""

    ar_emb = _emb_model.encode(
        [vietnamese_question], normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    sims = val_embeddings @ ar_emb

    if english_question:
        en_emb = _emb_model.encode(
            [english_question], normalize_embeddings=True,
            show_progress_bar=False,
        )[0]
        en_sims = val_embeddings @ en_emb
        sims = np.maximum(sims, en_sims)

    top_indices = np.argsort(-sims)[:top_k]
    hints = []
    seen_vals = set()
    for idx in top_indices:
        if sims[idx] < threshold:
            break
        val = values[idx]
        if val.lower() in seen_vals:
            continue
        seen_vals.add(val.lower())
        col_ref = col_refs[idx]
        hints.append(f"  → {col_ref} = '{val}'")

    if not hints:
        return ""
    return "Detected values in question (use EXACT spelling in WHERE):\n" + "\n".join(hints)


# ── Sample rows in prompt ───────────────────────────────────────

def get_sample_rows(db_path: str, schema_ddl: str, column_hints: str = "",
                     limit: int = 3, max_tables: int = 3) -> str:
    """Get sample rows from the most relevant tables."""
    if not db_path or not os.path.exists(db_path):
        return ""
    schema_tables = extract_tables_columns(schema_ddl)
    if not schema_tables:
        return ""

    priority_tables = []
    if column_hints:
        for tbl in schema_tables:
            if tbl in column_hints:
                priority_tables.append(tbl)
    for tbl in schema_tables:
        if tbl not in priority_tables:
            priority_tables.append(tbl)
    selected_tables = priority_tables[:max_tables]

    lines = []
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=10)
        conn.text_factory = str
        cursor = conn.cursor()
        for tbl in selected_tables:
            cols = schema_tables.get(tbl, [])
            if not cols:
                continue
            try:
                display_cols = cols[:6]
                col_str = ', '.join(f'"{c}"' for c in display_cols)
                cursor.execute(f'SELECT {col_str} FROM "{tbl}" LIMIT {limit}')
                rows = cursor.fetchall()
                if not rows:
                    continue
                header = " | ".join(c[:15] for c in display_cols)
                lines.append(f"  {tbl}: {header}")
                for row in rows:
                    row_str = " | ".join(
                        str(v)[:15] if v is not None else "NULL"
                        for v in row
                    )
                    lines.append(f"    {row_str}")
            except Exception:
                continue
        conn.close()
    except Exception:
        return ""

    if not lines:
        return ""
    return "Sample data (actual rows from database):\n" + "\n".join(lines)

#@title 🧰 Shared: Prompt Templates & Builders { display-mode: "form" }

# ── Prompt templates (shared between training and inference) ────

TIER1_SYSTEM = SQL_ONLY_SYSTEM  # reuse the proven system prompt

TIER1_PROMPT = '''Database Engine: SQLite

Database Schema:
{schema}

{value_hints}
{matched_values}
{column_hints}
{sample_rows}
Table -> Column Reference (use ONLY columns from the table that has them):
{table_summary}

IMPORTANT: Use the MINIMUM number of tables needed.
If ALL required columns exist in ONE table, do NOT use JOIN.
Only JOIN tables when the question requires data from multiple tables.
Use the exact column values shown above when filtering.

Question:
{question}
{english_hint}
Generate the SQL query. Output ONLY SQL inside a code block:
```sql
```'''

TIER1_RETRY_PROMPT = '''Database Engine: SQLite

Database Schema:
{schema}

{value_hints}
Your previous SQL had an error:
  SQL: {failed_sql}
  Error: {error_msg}

Question:
{question}

Fix the SQL. Output ONLY the corrected SQL inside a code block:
```sql
```'''





# ── Inference prompt builder (unchanged logic) ──────────────────



def _question_has_text_entities(question: str, english_question: str = "") -> bool:
    """Check if the question likely needs text value matching.
    Returns False for pure numeric/count/aggregation queries where
    value injection would just add noise.

    Heuristic: True if the question contains proper nouns, quoted strings,
    or named entities that could match DB text values."""
    combined = question + " " + english_question

    # Check for quoted strings in the question
    if re.search(r'["\'].+?["\'"]', combined):
        return True

    # Check for Latin proper nouns (capitalized words that aren't SQL keywords)
    sql_words = {'select','from','where','join','and','or','not','in','the',
                 'what','which','how','many','show','find','list','give','all',
                 'is','are','was','were','has','have','had','does','did',
                 'who','when','with','for','that','this','than','then',
                 'more','less','each','every','any','some','no','by',
                 'on','at','to','of','a','an','do','be','it','its',
                 'count','average','maximum','minimum','total','number',
                 'most','least','highest','lowest','largest','smallest',
                 'between','above','below','greater','smaller','equal',
                 'name','names','date','dates','id','type','types',
                 'started','ended','released','called','located',
                 'higher','lower','different','same','both','either'}
    # Vietnamese common words that appear capitalized at sentence start
    # These are NOT proper nouns — exclude from entity detection
    vietnamese_common_words = {
        'cho', 'tìm', 'hiển', 'thị', 'liệt', 'kê', 'tên', 'các',
        'những', 'của', 'trong', 'nào', 'bao', 'nhiêu', 'mấy',
        'được', 'bởi', 'theo', 'với', 'giữa', 'trên', 'dưới',
        'không', 'phải', 'hay', 'hoặc', 'và', 'nhưng', 'mà',
        'đã', 'đang', 'sẽ', 'là', 'có', 'bị', 'nhất',
        'lớn', 'nhỏ', 'cao', 'thấp', 'nhiều', 'ít',
        'trung', 'bình', 'tổng', 'cộng', 'số', 'lượng',
        'ngày', 'tháng', 'năm', 'giờ', 'phút',
        'hãy', 'xin', 'vui', 'lòng', 'biết',
        'cấp', 'bậc', 'loại', 'dân', 'trạm', 'thành', 'phố',
        'trả', 'về', 'kết', 'quả', 'giá', 'trị',
        'từ', 'đến', 'qua', 'sau', 'trước',
        'một', 'hai', 'ba', 'bốn', 'năm', 'sáu', 'bảy', 'tám', 'chín', 'mười',
        'toàn', 'bộ', 'riêng', 'biệt', 'khác', 'nhau',
        'shows', 'indicates', 'displays', 'find', 'list',
    }
    exclude_words = sql_words | vietnamese_common_words
    latin_words = re.findall(r'\b([A-Z][a-z]{2,})\b', combined)
    proper_nouns = [w for w in latin_words if w.lower() not in exclude_words]
    if len(proper_nouns) >= 1:
        return True

    # Check for Vietnamese entity markers — proper nouns often capitalized
    # Vietnamese uses Latin script so proper nouns are easier to detect.
    # Check if the question contains proper nouns or foreign names
    # Vietnamese proper nouns and foreign names are typically capitalized
    # Simple heuristic: if there are capitalized Vietnamese words that don't
    # appear in a basic Vietnamese SQL vocabulary, likely an entity
    vietnamese_sql_vocab = {
        'gì', 'nào', 'bao', 'nhiêu', 'mấy', 'ai', 'đâu', 'là', 'của',
        'và', 'hoặc', 'hay', 'không', 'tất', 'cả', 'mỗi', 'số', 'tên',
        'hiển', 'thị', 'tìm', 'cho', 'lớn', 'nhỏ', 'cao', 'thấp',
        'ít', 'nhiều', 'trung', 'bình', 'tổng', 'cộng', 'mà', 'cái',
        'được', 'có', 'bị', 'ở', 'trong', 'trên', 'dưới', 'giữa',
        'từ', 'đến', 'với', 'theo', 'về', 'cho', 'biết', 'liệt', 'kê',
        'ngày', 'tháng', 'năm', 'thời', 'gian', 'lúc',
    }

    # If numeric-only WHERE is likely (question has numbers but no text entities)
    has_numbers = bool(re.search(r'\b\d+\b', combined))
    has_comparison = bool(re.search(
        r'(lớn hơn|nhỏ hơn|cao hơn|thấp hơn|nhiều hơn|ít hơn|greater|less|more|than|higher|lower|above|below|between|>|<|>=|<=)',
        combined, re.I))

    # Pure numeric comparison with no proper nouns → skip
    if has_numbers and has_comparison and not proper_nouns:
        return False

    # Very short questions that are just counts → skip
    count_patterns = ['bao nhiêu', 'how many', 'count', 'số lượng', 'mấy']
    q_lower = combined.lower()
    if any(p in q_lower for p in count_patterns):
        # Count query — only inject if there's a named entity
        if not proper_nouns and not re.search(r'["\'"].+?["\'"]', combined):
            return False

    # Default: inject (safer to include than exclude)
    return True

def format_tier1_prompt(sample, use_ddl_override: bool = False) -> str:
    """Build enhanced prompt with all features (no dropout). Used at inference time."""
    if ENABLE_MSCHEMA and not use_ddl_override:
        schema_str = get_mschema(sample.db_id, sample.schema_ddl)
    else:
        schema_str = sample.schema_ddl[:4500]

    table_summary = build_table_column_summary(sample.schema_ddl)

    value_hints = ""
    if ENABLE_VALUE_HINTS and sample.db_path:
        value_hints = sample_db_values_question_aware(
            sample.db_path, sample.schema_ddl,
            sample.vietnamese_question, max_per_col=VALUE_HINT_MAX_PER_COL)

    column_hints = ""
    if ENABLE_COLUMN_LINKING:
        column_hints = link_columns_to_question(
            sample.db_id, sample.schema_ddl,
            sample.vietnamese_question,
            db_path=sample.db_path,
            top_k=COL_LINK_TOP_K)

    # V5 fix: skip value injection for non-entity questions (numeric/count queries)
    matched_values = ""
    if ENABLE_VALUE_INJECTION and sample.db_path:
        if _question_has_text_entities(sample.vietnamese_question, sample.english_question):
            matched_values = find_question_matched_values(
                sample.db_id, sample.db_path, sample.schema_ddl,
                sample.vietnamese_question, sample.english_question,
                top_k=VALUE_INJECTION_TOP_K, threshold=VALUE_INJECTION_THRESHOLD)

    sample_rows = ""
    if ENABLE_SAMPLE_ROWS and sample.db_path:
        sample_rows = get_sample_rows(
            sample.db_path, sample.schema_ddl, column_hints,
            limit=SAMPLE_ROWS_LIMIT, max_tables=SAMPLE_ROWS_MAX_TABLES)

    # FIX 4: Always include English hint (no dropout)
    english_hint = ""
    if sample.english_question:
        english_hint = f"(English: {sample.english_question})\n"

    return TIER1_PROMPT.format(
        schema=schema_str, value_hints=value_hints,
        matched_values=matched_values,
        column_hints=column_hints,
        sample_rows=sample_rows,
        table_summary=table_summary, question=sample.vietnamese_question,
        english_hint=english_hint)


# ── Training prompt builder (with feature dropout) ──────────────

import random

def format_training_prompt(sample, dropout_rate: float = 0.20) -> str:
    """
    Build training prompt ALIGNED with inference format, but with
    random feature dropout for robustness.

    Each feature is independently dropped with probability = dropout_rate.
    This teaches the model to USE features when present but not DEPEND
    on any single one — graceful degradation if a hint is noisy or absent.
    """
    # Schema: M-Schema or fallback to DDL
    if ENABLE_MSCHEMA and random.random() >= dropout_rate:
        schema_str = get_mschema(sample.db_id, sample.schema_ddl)
    else:
        schema_str = sample.schema_ddl[:4500]

    table_summary = build_table_column_summary(sample.schema_ddl)

    # Value hints
    value_hints = ""
    if (ENABLE_VALUE_HINTS and sample.db_path
            and random.random() >= dropout_rate):
        value_hints = sample_db_values_question_aware(
            sample.db_path, sample.schema_ddl,
            sample.vietnamese_question, max_per_col=VALUE_HINT_MAX_PER_COL)

    # Column linking (requires embedding model)
    # Higher dropout (40%) since column linking is the noisiest feature
    column_hints = ""
    if (ENABLE_COLUMN_LINKING and _emb_model is not None
            and random.random() >= 0.40):
        column_hints = link_columns_to_question(
            sample.db_id, sample.schema_ddl,
            sample.vietnamese_question,
            db_path=sample.db_path,
            top_k=COL_LINK_TOP_K)

    # Question-matched value injection (requires embedding model)
    # V5 fix: skip for non-entity questions (numeric/count queries)
    matched_values = ""
    if (ENABLE_VALUE_INJECTION and sample.db_path
            and _emb_model is not None
            and random.random() >= dropout_rate
            and _question_has_text_entities(sample.vietnamese_question, sample.english_question)):
        matched_values = find_question_matched_values(
            sample.db_id, sample.db_path, sample.schema_ddl,
            sample.vietnamese_question, sample.english_question,
            top_k=VALUE_INJECTION_TOP_K, threshold=VALUE_INJECTION_THRESHOLD)

    # Sample rows
    sample_rows = ""
    if (ENABLE_SAMPLE_ROWS and sample.db_path
            and random.random() >= dropout_rate):
        sample_rows = get_sample_rows(
            sample.db_path, sample.schema_ddl, column_hints,
            limit=SAMPLE_ROWS_LIMIT, max_tables=SAMPLE_ROWS_MAX_TABLES)

    # English hint — ALWAYS included (no dropout) per FIX 4
    english_hint = ""
    if sample.english_question:
        english_hint = f"(English: {sample.english_question})\n"

    return TIER1_PROMPT.format(
        schema=schema_str, value_hints=value_hints,
        matched_values=matched_values,
        column_hints=column_hints,
        sample_rows=sample_rows,
        table_summary=table_summary, question=sample.vietnamese_question,
        english_hint=english_hint)


def prewarm_column_cache():
    """Pre-compute column embeddings for all databases."""
    if _emb_model is None:
        print("⚠️ Embedding model not loaded — skipping column cache")
        return
    count = 0
    for db_id, schema_ddl in multispider_vi_schemas.items():
        _get_column_embeddings(db_id, schema_ddl)
        count += 1
    total_cols = sum(len(v['candidates']) for v in _col_emb_cache.values())
    print(f"✅ Column embeddings cached for {count} databases ({total_cols} columns)")


def prewarm_value_cache():
    """Pre-compute value embeddings for all databases with SQLite files."""
    if _emb_model is None:
        print("⚠️ Embedding model not loaded — skipping value cache")
        return
    count = 0
    total_vals = 0
    for db_id, schema_ddl in multispider_vi_schemas.items():
        db_path = db_paths.get(db_id, "")
        if db_path:
            data = _build_value_embeddings(db_id, db_path, schema_ddl)
            total_vals += len(data["values"])
            count += 1
    print(f"✅ Value embeddings cached for {count} databases ({total_vals} unique values)")


print("✅ Shared functions defined (M-Schema, column linking, value hints, "
      "sample rows, prompt builders)")


---
## 🇻🇳 Section C0: Generate Vietnamese Table Glosses & Column Descriptions

Extracts Vietnamese table/column names from MultiSpider's `tables_vi.json` files by comparing them with the English `tables.json`. Saves to Drive cache for reuse.

In [ ]:

#@title 🇻🇳 Generate Vietnamese Descriptions from MultiSpider tables_vi.json { display-mode: "form" }

# ═══════════════════════════════════════════════════════════════════
# MultiSpider ships per-database tables_vi.json files containing
# Vietnamese translations of table names and column names.
#
# Spider tables.json format:
#   table_names_original: ["department", "head"]       ← English originals
#   table_names:          ["phòng ban", "người đứng đầu"]  ← Vietnamese in tables_vi.json
#   column_names_original: [[0, "Department_ID"], ...]
#   column_names:          [[0, "mã phòng ban"], ...]  ← Vietnamese in tables_vi.json
#
# We extract these and build the pipeline dictionaries:
#   _vi_table_glosses   = {db_id: {"department": "phòng ban", ...}}
#   _vi_col_descriptions = {db_id: {"department.Department_ID": "mã phòng ban", ...}}
# ═══════════════════════════════════════════════════════════════════

import json, glob, os

HF_LOCAL = "/content/hf_multispider"
MULTISPIDER_DIR = "/content/multispider"

# ── 1. Load the main English tables.json (already copied in download cell) ──
TABLES_JSON_PATH = os.path.join(MULTISPIDER_DIR, "tables.json")
with open(TABLES_JSON_PATH, "r", encoding="utf-8") as f:
    tables_en = json.load(f)

# Index by db_id for fast lookup
tables_en_by_id = {entry["db_id"]: entry for entry in tables_en}
print(f"✅ Loaded English tables.json: {len(tables_en_by_id)} databases")

# ── 2. Find Vietnamese tables_vi.json files ──────────────────────
# They can be in: dataset/spider/database/<db_name>/tables_vi.json
# or: dataset/multispider/*/tables_vi.json (centralized)
vi_table_files = sorted(glob.glob(os.path.join(HF_LOCAL, "**", "tables_vi.json"), recursive=True))
print(f"📂 Found {len(vi_table_files)} tables_vi.json files")

# Load and merge all Vietnamese table definitions
tables_vi_by_id = {}
for vf in vi_table_files:
    try:
        with open(vf, "r", encoding="utf-8") as f:
            data = json.load(f)
        # Could be a list of entries or a single entry
        if isinstance(data, list):
            for entry in data:
                if "db_id" in entry:
                    tables_vi_by_id[entry["db_id"]] = entry
        elif isinstance(data, dict) and "db_id" in data:
            tables_vi_by_id[data["db_id"]] = data
    except Exception as e:
        print(f"   ⚠️ Failed to load {os.path.relpath(vf, HF_LOCAL)}: {e}")

print(f"✅ Loaded Vietnamese table definitions for {len(tables_vi_by_id)} databases")

# ── 3. Build Vietnamese table glosses ─────────────────────────────
# Format: {db_id: {"english_table_name": "Vietnamese gloss"}}
_vi_table_glosses_generated = {}
total_glosses = 0

for db_id, vi_entry in tables_vi_by_id.items():
    en_entry = tables_en_by_id.get(db_id)
    if not en_entry:
        continue

    en_tables = en_entry.get("table_names_original", en_entry.get("table_names", []))
    vi_tables = vi_entry.get("table_names", [])

    if len(en_tables) != len(vi_tables):
        continue

    glosses = {}
    for en_name, vi_name in zip(en_tables, vi_tables):
        # Only add if Vietnamese differs from English (actual translation)
        if vi_name and vi_name.lower() != en_name.lower():
            glosses[en_name] = vi_name
            total_glosses += 1

    if glosses:
        _vi_table_glosses_generated[db_id] = glosses

print(f"✅ Generated {total_glosses} Vietnamese table glosses across {len(_vi_table_glosses_generated)} databases")

# Show examples
shown = 0
for db_id, glosses in list(_vi_table_glosses_generated.items())[:5]:
    for en, vi in list(glosses.items())[:3]:
        print(f"   {db_id}: {en} → {vi}")
        shown += 1
    if shown > 10:
        break

# ── 4. Build Vietnamese column descriptions ──────────────────────
# Format: {db_id: {"table.column": "Vietnamese description"}}
_vi_col_descriptions_generated = {}
total_descs = 0

for db_id, vi_entry in tables_vi_by_id.items():
    en_entry = tables_en_by_id.get(db_id)
    if not en_entry:
        continue

    en_tables = en_entry.get("table_names_original", en_entry.get("table_names", []))
    en_cols = en_entry.get("column_names_original", en_entry.get("column_names", []))
    vi_cols = vi_entry.get("column_names", [])

    if len(en_cols) != len(vi_cols):
        continue

    descs = {}
    for (en_tbl_idx, en_col_name), (vi_tbl_idx, vi_col_name) in zip(en_cols, vi_cols):
        if en_tbl_idx < 0 or vi_tbl_idx < 0:
            continue  # skip the wildcard "*" entry
        if en_tbl_idx >= len(en_tables):
            continue

        en_table = en_tables[en_tbl_idx]
        # Only add if Vietnamese differs from English
        if vi_col_name and vi_col_name.lower() != en_col_name.lower():
            descs[f"{en_table}.{en_col_name}"] = vi_col_name
            total_descs += 1

    if descs:
        _vi_col_descriptions_generated[db_id] = descs

print(f"✅ Generated {total_descs} Vietnamese column descriptions across {len(_vi_col_descriptions_generated)} databases")

# Show examples
shown = 0
for db_id, descs in list(_vi_col_descriptions_generated.items())[:5]:
    for en, vi in list(descs.items())[:3]:
        print(f"   {db_id}: {en} → {vi}")
        shown += 1
    if shown > 10:
        break

# ── 5. Fill gaps with Google Translate ────────────────────────────
# MultiSpider tables_vi.json covers most databases, but some columns
# have identical English/Vietnamese names (technical terms, IDs, etc.)
# and some databases may not have tables_vi.json at all.
# Use Google Translate to fill these gaps.

FILL_GAPS_WITH_TRANSLATE = True  #@param {type:"boolean"}
TRANSLATE_CACHE_PATH = "/content/drive/MyDrive/vi_translate_cache.json"  #@param {type:"string"}

if FILL_GAPS_WITH_TRANSLATE:
    # ── Load translator ──
    try:
        from deep_translator import GoogleTranslator
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "deep-translator"], check=True)
        from deep_translator import GoogleTranslator

    _gap_translator = GoogleTranslator(source='en', target='vi')

    # ── Load translation cache from Drive ──
    _translate_cache = {}
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists(TRANSLATE_CACHE_PATH):
        try:
            with open(TRANSLATE_CACHE_PATH, "r", encoding="utf-8") as f:
                _translate_cache = json.load(f)
            print(f"📂 Loaded {len(_translate_cache)} cached translations")
        except Exception:
            pass

    def _translate_en_to_vi(text):
        """Translate English text to Vietnamese with caching."""
        if text in _translate_cache:
            return _translate_cache[text]
        try:
            result = _gap_translator.translate(text)
            if result and len(result.strip()) > 0:
                _translate_cache[text] = result.strip()
                return result.strip()
        except Exception:
            import time
            time.sleep(0.3)
            try:
                result = _gap_translator.translate(text)
                if result:
                    _translate_cache[text] = result.strip()
                    return result.strip()
            except Exception:
                pass
        return ""

    def _make_readable(col_name):
        """Convert CamelCase/snake_case column name to readable English."""
        # CamelCase → spaced: "Department_ID" → "Department ID"
        s = re.sub(r'([a-z])([A-Z])', r'\1 \2', col_name)
        s = s.replace('_', ' ')
        # Remove "id" suffix for cleaner translation
        # But keep it if the column is literally just "id"
        return s.strip()

    # ── 5a. Fill missing table glosses ──
    tables_filled = 0
    for db_id, schema_ddl in multispider_vi_schemas.items():
        if db_id not in _vi_table_glosses_generated:
            _vi_table_glosses_generated[db_id] = {}

        schema_tables = extract_tables_columns(schema_ddl)
        for table_name in schema_tables:
            if table_name not in _vi_table_glosses_generated.get(db_id, {}):
                readable = _make_readable(table_name)
                vi = _translate_en_to_vi(readable)
                if vi and vi.lower() != readable.lower():
                    _vi_table_glosses_generated.setdefault(db_id, {})[table_name] = vi
                    tables_filled += 1

    print(f"🌐 Google Translate filled {tables_filled} missing table glosses")

    # ── 5b. Fill missing column descriptions ──
    cols_filled = 0
    for db_id, schema_ddl in multispider_vi_schemas.items():
        if db_id not in _vi_col_descriptions_generated:
            _vi_col_descriptions_generated[db_id] = {}

        schema_tables = extract_tables_columns(schema_ddl)
        for table_name, columns in schema_tables.items():
            for col_name in columns:
                key = f"{table_name}.{col_name}"
                if key not in _vi_col_descriptions_generated.get(db_id, {}):
                    readable = _make_readable(col_name)
                    # Add table context for ambiguous columns
                    if readable.lower() in ('id', 'name', 'type', 'code', 'date', 'status'):
                        readable = f"{_make_readable(table_name)} {readable}"
                    vi = _translate_en_to_vi(readable)
                    if vi and vi.lower() != readable.lower():
                        _vi_col_descriptions_generated.setdefault(db_id, {})[key] = vi
                        cols_filled += 1

        # Progress
        if cols_filled > 0 and cols_filled % 500 == 0:
            print(f"   [{cols_filled} columns translated...]")

    print(f"🌐 Google Translate filled {cols_filled} missing column descriptions")

    # ── 5c. Save translation cache ──
    os.makedirs(os.path.dirname(TRANSLATE_CACHE_PATH), exist_ok=True)
    with open(TRANSLATE_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(_translate_cache, f, ensure_ascii=False, indent=2)
    print(f"💾 Translation cache saved: {len(_translate_cache)} entries → {TRANSLATE_CACHE_PATH}")

    # Show examples of gap-filled translations
    print(f"\n📋 Gap-filled translation examples:")
    shown = 0
    for db_id, descs in list(_vi_col_descriptions_generated.items()):
        for key, vi in list(descs.items()):
            en_col = key.split(".")[-1]
            if _make_readable(en_col) in _translate_cache and shown < 8:
                print(f"   {key} → {vi}")
                shown += 1
        if shown >= 8:
            break

    total_glosses_now = sum(len(v) for v in _vi_table_glosses_generated.values())
    total_descs_now = sum(len(v) for v in _vi_col_descriptions_generated.values())
    print(f"\n✅ After gap-filling:")
    print(f"   Table glosses:       {total_glosses_now} (MultiSpider + Google Translate)")
    print(f"   Column descriptions: {total_descs_now} (MultiSpider + Google Translate)")

else:
    print("ℹ️  Google Translate gap-filling disabled")

# ── 6. Save to Drive cache paths (so Section C can load them) ────
from google.colab import drive
drive.mount('/content/drive')

# Save table glosses
if _vi_table_glosses_generated:
    os.makedirs(os.path.dirname(VI_TABLE_GLOSS_PATH), exist_ok=True)
    with open(VI_TABLE_GLOSS_PATH, "w", encoding="utf-8") as f:
        json.dump(_vi_table_glosses_generated, f, ensure_ascii=False, indent=2)
    print(f"\n💾 Saved Vietnamese table glosses → {VI_TABLE_GLOSS_PATH}")

# Save column descriptions
if _vi_col_descriptions_generated:
    os.makedirs(os.path.dirname(VI_COL_DESC_PATH), exist_ok=True)
    with open(VI_COL_DESC_PATH, "w", encoding="utf-8") as f:
        json.dump(_vi_col_descriptions_generated, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved Vietnamese column descriptions → {VI_COL_DESC_PATH}")

# ── 7. Also populate the in-memory dictionaries directly ──────────
# (So they're available even if Section C loading from file fails)
_vi_table_glosses.update(_vi_table_glosses_generated)
_vi_col_descriptions.update(_vi_col_descriptions_generated)

print(f"\n✅ Vietnamese bilingual annotations ready:")
print(f"   Table glosses:       {sum(len(v) for v in _vi_table_glosses.values())} across {len(_vi_table_glosses)} databases")
print(f"   Column descriptions: {sum(len(v) for v in _vi_col_descriptions.values())} across {len(_vi_col_descriptions)} databases")
print(f"   M-Schema will now show: 【department】 (phòng ban)")
print(f"   Column hints will show: ★ head.age — tuổi")


---
## 🔧 Section C: Pre-Training Resource Loading

Loads embedding model + Vietnamese caches from Drive. Pre-warms embedding caches.

**VRAM note:** BGE-M3 uses ~2.3 GB. On A100 (40 GB) this coexists easily with QLoRA training.

In [ ]:

# Paste as a NEW CELL after Section B.
# Loads the embedding model, Vietnamese description caches, and
# pre-warms the embedding caches before training data construction.
# ═══════════════════════════════════════════════════════════════════



#@title 🔧 Pre-Training Resource Loading { display-mode: "form" }

from google.colab import drive
drive.mount('/content/drive')

# ── 1. Load Vietnamese column descriptions from cache ──────────────

if ENABLE_VI_COL_DESC and os.path.exists(VI_COL_DESC_PATH):
    try:
        with open(VI_COL_DESC_PATH, "r", encoding="utf-8") as f:
            _vi_col_descriptions = json.load(f)
        total_descs = sum(len(v) for v in _vi_col_descriptions.values())
        print(f"✅ Loaded {total_descs} Vietnamese column descriptions from cache")
    except Exception as e:
        print(f"⚠️ Failed to load column descriptions: {e}")
        print(f"   Column linking will work without Vietnamese descriptions.")
elif ENABLE_VI_COL_DESC:
    print(f"⚠️ Vietnamese column descriptions not found at: {VI_COL_DESC_PATH}")
    print(f"   They will be generated later using the fine-tuned model.")
    print(f"   For now, training prompts will use English-only column descriptions.")

# ── 2. Load Vietnamese table glosses from cache ────────────────────

if ENABLE_BILINGUAL_SCHEMA and os.path.exists(VI_TABLE_GLOSS_PATH):
    try:
        with open(VI_TABLE_GLOSS_PATH, "r", encoding="utf-8") as f:
            _vi_table_glosses = json.load(f)
        total_glosses = sum(len(v) for v in _vi_table_glosses.values())
        print(f"✅ Loaded {total_glosses} Vietnamese table glosses from cache")
    except Exception as e:
        print(f"⚠️ Failed to load table glosses: {e}")
elif ENABLE_BILINGUAL_SCHEMA:
    print(f"⚠️ Table glosses not found at: {VI_TABLE_GLOSS_PATH}")
    print(f"   M-Schema will use English-only table names for training.")

# ── 3. Load embedding model ────────────────────────────────────
#    ~2.3 GB VRAM.  On A100 (40GB): easily coexists with QLoRA training.
#    On T4 (16GB): tight — set TRAIN_LOAD_EMBEDDINGS = False to skip.

TRAIN_LOAD_EMBEDDINGS = True  #@param {type:"boolean"}

if TRAIN_LOAD_EMBEDDINGS and (ENABLE_COLUMN_LINKING or ENABLE_VALUE_INJECTION):
    from sentence_transformers import SentenceTransformer
    print("🔄 Loading BGE-M3 embedding model for training prompt construction...")
    _emb_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
    print(f"✅ BGE-M3 loaded ({_emb_model.get_sentence_embedding_dimension()}d, ~2.3GB VRAM)")

    # Pre-warm caches (one-time cost, ~2-5 min)
    print("🔄 Pre-warming embedding caches for all databases...")
    prewarm_column_cache()
    prewarm_value_cache()
else:
    print("ℹ️  Embedding model not loaded for training.")
    print("   Column linking and value injection will be SKIPPED in training prompts.")
    print("   Other features (M-Schema, value hints, English hint, sample rows) still active.")

print()
print("🎯 Pre-training resources ready. Training prompts will use:")
print(f"   M-Schema with bilingual annotations : {'✅' if ENABLE_MSCHEMA and _vi_table_glosses else '⚠️ English-only' if ENABLE_MSCHEMA else '❌'}")
print(f"   Column linking (embedding)          : {'✅' if _emb_model is not None and ENABLE_COLUMN_LINKING else '❌'}")
print(f"   Vietnamese column descriptions          : {'✅' if _vi_col_descriptions else '❌'}")
print(f"   Value hints from DB                 : {'✅' if ENABLE_VALUE_HINTS else '❌'}")
print(f"   Question-matched value injection    : {'✅' if _emb_model is not None and ENABLE_VALUE_INJECTION else '❌'}")
print(f"   Sample rows                         : {'✅' if ENABLE_SAMPLE_ROWS else '❌'}")
print(f"   English translation hint            : {'✅' if ENABLE_ENGLISH_HINT else '❌'}")
print(f"   Feature dropout rate                : {FEATURE_DROPOUT_RATE}")


---
## 🌌 Translate Training Questions to English

> Uses Google Translate with Drive caching. First run: ~3-5 min for ~7000 samples. Subsequent runs: instant from cache.

---
## 📂 Section D: Load train.json → Training Data (Prompt-Aligned)

Uses the aligned Tier 1 prompt format with feature dropout.

In [ ]:

#@title 🌌 Translate Training Questions (Google Translate + Cache) { display-mode: "form" }

TRAIN_ENGLISH_CACHE = "/content/drive/MyDrive/vi_multispider_train_english.json"  #@param {type:"string"}
REGENERATE_TRAIN_TRANSLATIONS = False  #@param {type:"boolean"}

def _is_vietnamese(text: str) -> bool:
    """Check if text contains Vietnamese diacritical characters."""
    if not text:
        return False
    # Vietnamese uses Latin script with specific diacritics
    vietnamese_chars = set("àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ"
                          "ÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬÈÉẺẼẸÊỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴĐ")
    vi_count = sum(1 for c in text if c in vietnamese_chars)
    return vi_count > len(text) * 0.03  # Vietnamese has fewer diacritics per char compared to Arabic script density

_translator = None

def _load_translator():
    global _translator
    if _translator is not None:
        return
    try:
        from deep_translator import GoogleTranslator
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "deep-translator"], check=True)
        from deep_translator import GoogleTranslator
    _translator = GoogleTranslator(source='vi', target='en')
    print("✅ Google Translate loaded")


def _translate_vietnamese_to_english(vietnamese_text: str) -> str:
    """Translate Vietnamese to English using Google Translate."""
    try:
        result = _translator.translate(vietnamese_text)
        return result.strip() if result and len(result.strip()) > 3 else ""
    except Exception:
        import time
        time.sleep(0.5)
        try:
            result = _translator.translate(vietnamese_text)
            return result.strip() if result and len(result.strip()) > 3 else ""
        except Exception:
            return ""

print("✅ Translation functions defined")
print("   ℹ️ Translations will be applied after training data is loaded (Section D)")

# This REPLACES your existing "📂 Load train.json → Training Data"

def _is_vietnamese(text: str) -> bool:
    """Check if text contains Vietnamese diacritical characters."""
    if not text:
        return False
    # Vietnamese uses Latin script with specific diacritics
    vietnamese_chars = set("àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ"
                          "ÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬÈÉẺẼẸÊỀẾỂỄỆÌÍỈĨỊÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢÙÚỦŨỤƯỪỨỬỮỰỲÝỶỸỴĐ")
    vi_count = sum(1 for c in text if c in vietnamese_chars)
    return vi_count > len(text) * 0.03  # Vietnamese has fewer diacritics per char compared to Arabic script density

# cell.  Uses the aligned Tier 1 prompt with feature dropout.
# ═══════════════════════════════════════════════════════════════════



#@title 📂 Load train.json → Training Data (Prompt-Aligned) { display-mode: "form" }

print("📂 Loading MultiSpider-Vietnamese train.json...")
with open(os.path.join(MULTISPIDER_DIR, "train.json"), "r", encoding="utf-8") as f:
    train_raw = json.load(f)
print(f"   Raw samples: {len(train_raw)}")

# ── Build training samples as EvalSample objects ────────────────
print(f"\n🧹 Building training data...")
if TRAIN_USE_ALIGNED_PROMPT:
    print(f"   ✅ PROMPT ALIGNMENT ON — using Tier 1 format with feature dropout ({FEATURE_DROPOUT_RATE})")
else:
    print(f"   ℹ️  Prompt alignment OFF — using base format (DDL + table summary)")

training_data = []
train_samples = []  # EvalSample objects for preview
skipped = {"no_schema": 0, "no_sql": 0}

random.seed(42)  # reproducible dropout for this construction pass

for idx, r in enumerate(train_raw):
    db_id = r["db_id"]
    schema = multispider_vi_schemas.get(db_id, "")
    vietnamese_question = r.get("Vietnamese", r.get("question_tgt", r.get("question", "")))
    # Detect if "question" field is Vietnamese (check for Vietnamese diacritics)
    _raw_q = r.get("question", "")
    english_question = "" if _is_vietnamese(_raw_q) else _raw_q
    sql = r.get("query", "").strip()

    if not schema:
        skipped["no_schema"] += 1
        continue
    if not sql or not re.search(r'\bSELECT\b', sql, re.I):
        skipped["no_sql"] += 1
        continue

    # Build EvalSample for this training example
    sample = EvalSample(
        id=f"train_{idx:04d}",
        db_id=db_id,
        vietnamese_question=vietnamese_question,
        gold_sql=sql,
        schema_ddl=schema,
        db_path=db_paths.get(db_id, ""),
        english_question=english_question,
    )
    train_samples.append(sample)

    # Build prompt — aligned with inference or base format
    if TRAIN_USE_ALIGNED_PROMPT:
        prompt = format_training_prompt(sample, dropout_rate=FEATURE_DROPOUT_RATE)
    else:
        prompt = format_prompt(schema, vietnamese_question)

    training_data.append({
        "input": prompt,
        "output": f"```sql\n{sql}\n```",
        "sql": sql,
        "question": vietnamese_question,
        "db_id": db_id,
    })

    # Progress
    if (idx + 1) % 1000 == 0:
        print(f"   [{idx + 1}/{len(train_raw)}] processed...")

print(f"\n✅ Training set: {len(training_data)} examples")
print(f"   Skipped: {skipped['no_schema']} no schema, {skipped['no_sql']} no SQL")
print(f"   Unique databases: {len(set(d['db_id'] for d in training_data))}")

# Show samples
for i, s in enumerate(training_data[:3]):
    print(f"\n   [{i+1}] Q: {s['question'][:70]}")
    print(f"       SQL: {s['sql'][:70]}")

# ── Token length statistics ─────────────────────────────────────
try:
    sample_lengths = []
    for d in training_data[:200]:
        # Rough estimate: 1 token ≈ 3.0 chars for mixed Vietnamese/English
        est_tokens = len(d["input"]) / 2.5 + len(d["output"]) / 2.5
        sample_lengths.append(est_tokens)
    avg_len = sum(sample_lengths) / len(sample_lengths)
    max_len = max(sample_lengths)
    over_limit = sum(1 for l in sample_lengths if l > MAX_SEQ_LENGTH_7B)
    print(f"\n   📊 Estimated token lengths (first 200):")
    print(f"      Avg: {avg_len:.0f}, Max: {max_len:.0f}")
    print(f"      Over MAX_SEQ_LENGTH ({MAX_SEQ_LENGTH_7B}): {over_limit}/{len(sample_lengths)}")
    if over_limit > 0:
        print(f"      ⚠️  {over_limit} examples may be truncated. Consider MAX_SEQ_LENGTH_7B = 4096.")
except Exception:
    pass

#@title 🌌 Apply English Translations to Training Data { display-mode: "form" }

# \u2500\u2500 Translate training questions \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
_train_english_cache = {}

if ENABLE_ENGLISH_HINT:
    # Try cache first
    cache_hit = False
    if os.path.exists(TRAIN_ENGLISH_CACHE) and not REGENERATE_TRAIN_TRANSLATIONS:
        try:
            with open(TRAIN_ENGLISH_CACHE, "r", encoding="utf-8") as f:
                _train_english_cache = json.load(f)
            print(f"📂 Loaded {len(_train_english_cache)} cached training translations")
            cache_hit = True
        except Exception as e:
            print(f"⚠️ Cache load failed: {e}")

    # Apply cached translations
    applied = 0
    missing_ids = []
    for s in train_samples:
        if s.id in _train_english_cache:
            eng = _train_english_cache[s.id]
            if eng and not _is_vietnamese(eng):
                s.english_question = eng
                applied += 1
            else:
                s.english_question = ""
                missing_ids.append(s)
        else:
            missing_ids.append(s)

    if cache_hit:
        print(f"   Applied {applied}/{len(train_samples)} from cache")

    # Translate missing samples
    if missing_ids:
        _load_translator()
        print(f"🌌 Translating {len(missing_ids)} missing training questions...")
        for i, s in enumerate(missing_ids):
            eng = _translate_vietnamese_to_english(s.vietnamese_question)
            if eng and not _is_vietnamese(eng):
                s.english_question = eng
                _train_english_cache[s.id] = eng
                applied += 1
            else:
                s.english_question = ""
                _train_english_cache[s.id] = ""
            if (i + 1) % 500 == 0:
                print(f"   [{i+1}/{len(missing_ids)}]")

        # Save updated cache
        os.makedirs(os.path.dirname(TRAIN_ENGLISH_CACHE), exist_ok=True)
        with open(TRAIN_ENGLISH_CACHE, "w", encoding="utf-8") as f:
            json.dump(_train_english_cache, f, ensure_ascii=False, indent=2)
        print(f"   ✅ Cache saved: {TRAIN_ENGLISH_CACHE}")

    print(f"\n✅ English translations applied: {applied}/{len(train_samples)}")

    # Show examples
    shown = 0
    for s in train_samples[:50]:
        if s.english_question and shown < 5:
            print(f"   {s.id}: VI: {s.vietnamese_question[:55]}")
            print(f"         EN: {s.english_question[:55]}")
            shown += 1
else:
    # Clear any Vietnamese text that leaked into english_question
    for s in train_samples:
        if _is_vietnamese(s.english_question):
            s.english_question = ""
    print("ℹ️  English hints disabled")

# \u2500\u2500 Rebuild training prompts with correct English \u2500\u2500\u2500\u2500\u2500\u2500\u2500
print(f"\n🔄 Rebuilding training prompts with correct English translations...")
random.seed(42)
rebuilt = 0
for i, (td, s) in enumerate(zip(training_data, train_samples)):
    if TRAIN_USE_ALIGNED_PROMPT:
        td["input"] = format_training_prompt(s, dropout_rate=FEATURE_DROPOUT_RATE)
    rebuilt += 1
print(f"✅ Rebuilt {rebuilt} training prompts")

# Verify no Vietnamese in English hints
source_lang_in_english = 0
for td in training_data[:200]:
    m = re.search(r'\(English: (.+?)\)', td["input"])
    if m and _is_vietnamese(m.group(1)):
        source_lang_in_english += 1
if source_lang_in_english > 0:
    print(f"⚠️ WARNING: {source_lang_in_english}/200 prompts still have Vietnamese in English hint!")
else:
    print(f"✅ Verified: 0/200 prompts have Vietnamese in English hint")


---
## 👁️ Section E: Preview — What the Training Model Sees

Color-coded prompts, feature statistics, and old-vs-new comparison.

In [ ]:

# Paste as a NEW CELL after Section D.
# Shows the full prompt for a few training examples so you can
# verify alignment with the inference prompt format.
# ═══════════════════════════════════════════════════════════════════



#@title 👁️ Preview: What the Training Model Sees { display-mode: "form" }

TRAIN_PREVIEW_IDS = [0, 50, 150, 200, 400, 800, 1200, 4000, 8000]  #@param {type:"raw"}

print("=" * 80)
print("  👁️ TRAINING PROMPT PREVIEW — What the model learns from")
print("=" * 80)

for sample_idx in TRAIN_PREVIEW_IDS:
    if sample_idx >= len(train_samples):
        continue
    sample = train_samples[sample_idx]

    # Build a FRESH prompt (with dropout) to show what training sees
    random.seed(sample_idx)  # deterministic for preview
    if TRAIN_USE_ALIGNED_PROMPT:
        prompt = format_training_prompt(sample, dropout_rate=FEATURE_DROPOUT_RATE)
    else:
        prompt = format_prompt(sample.schema_ddl, sample.vietnamese_question)

    # Estimate token count
    est_tokens = int(len(prompt) / 2.5)

    print(f"\n{'━' * 80}")
    print(f"  Sample: {sample.id} | {sample.db_id} | ~{est_tokens} est. tokens")
    print(f"  Question (VI): {sample.vietnamese_question[:100]}")
    print(f"  Question (EN): {sample.english_question[:100]}")
    print(f"  Gold SQL:      {sample.gold_sql[:100]}")
    print(f"{'━' * 80}")

    for line in prompt.split("\n"):
        if line.strip().startswith("★"):
            print(f"  \033[92m{line}\033[0m")           # green: column hints
        elif line.strip().startswith("→"):
            print(f"  \033[95m{line}\033[0m")           # magenta: value injection
        elif line.strip().startswith("Likely relevant"):
            print(f"  \033[92m{line}\033[0m")
        elif line.strip().startswith("Detected values"):
            print(f"  \033[95m{line}\033[0m")
        elif line.strip().startswith("Sample column values"):
            print(f"  \033[93m{line}\033[0m")           # yellow: value hints
        elif line.strip().startswith("Sample data"):
            print(f"  \033[93m{line}\033[0m")
        elif line.strip().startswith("Question:"):
            print(f"  \033[96m{line}\033[0m")           # cyan: question
        elif "(English:" in line:
            print(f"  \033[94m{line}\033[0m")           # blue: English hint
        elif line.strip().startswith("【"):
            print(f"  \033[91m{line}\033[0m")           # red: M-Schema tables
        else:
            print(f"  {line}")

    # Show the target (SQL output)
    print(f"\n  \033[1m  TARGET OUTPUT:\033[0m")
    print(f"  ```sql")
    print(f"  {sample.gold_sql}")
    print(f"  ```")
    print()

#@title 📊 Training Prompt Feature Statistics { display-mode: "form" }

# ── Feature presence statistics across training data ────────────
print("\n" + "=" * 80)
print("  📊 TRAINING PROMPT FEATURE STATISTICS (sampled from first 200 examples)")
print("=" * 80)

feature_counts = {
    "M-Schema (【)": 0,
    "Vietnamese table gloss": 0,
    "Vietnamese column desc": 0,
    "Value hints": 0,
    "Column linking (★)": 0,
    "Value injection (→)": 0,
    "Sample rows": 0,
    "English hint": 0,
}

n_check = min(200, len(training_data))
for i in range(n_check):
    p = training_data[i]["input"]
    if "【" in p:                           feature_counts["M-Schema (【)"] += 1
    # Vietnamese table gloss: 【table_name】 (Vietnamese gloss) — gloss is on SAME line, not column data
    if re.search(r'【\w+】\s*\([^*,)]+\)\s*$', p, re.MULTILINE):
                                            feature_counts["Vietnamese table gloss"] += 1
    if "—" in p and "★" in p:              feature_counts["Vietnamese column desc"] += 1
    if "Sample column values" in p:         feature_counts["Value hints"] += 1
    if "★" in p:                           feature_counts["Column linking (★)"] += 1
    if "→" in p and "Detected values" in p: feature_counts["Value injection (→)"] += 1
    if "Sample data" in p:                  feature_counts["Sample rows"] += 1
    if "(English:" in p:                    feature_counts["English hint"] += 1

print(f"  Checked {n_check} training examples:")
for feat, count in feature_counts.items():
    pct = 100 * count / n_check
    expected = 100 * (1 - FEATURE_DROPOUT_RATE)
    bar = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
    print(f"    {feat:<25s}: {count:>4d}/{n_check} ({pct:>5.1f}%) {bar}  [expected ~{expected:.0f}%]")

print(f"\n  💡 Each feature should appear in ~{100*(1-FEATURE_DROPOUT_RATE):.0f}% of examples "
      f"(dropout={FEATURE_DROPOUT_RATE})")
print(f"  💡 The model learns to USE features when present but not DEPEND on any single one.")

#@title 🔀 Prompt Comparison: Old vs New { display-mode: "form" }

# ── Side-by-side comparison: old vs new prompt ──────────────────
print("\n" + "=" * 80)
print("  🔀 PROMPT COMPARISON: Old (base) vs New (aligned)")
print("=" * 80)

if train_samples:
    compare_sample = train_samples[0]

    old_prompt = format_prompt(compare_sample.schema_ddl, compare_sample.vietnamese_question)
    random.seed(999)
    new_prompt = format_training_prompt(compare_sample, dropout_rate=0.0)  # no dropout for comparison

    old_chars = len(old_prompt)
    new_chars = len(new_prompt)

    print(f"\n  Sample: {compare_sample.id} | {compare_sample.db_id}")
    print(f"  Question: {compare_sample.vietnamese_question[:80]}")
    print(f"\n  Old prompt: {old_chars:,} chars (~{old_chars//3} tokens)")
    print(f"  New prompt: {new_chars:,} chars (~{new_chars//3} tokens)")
    print(f"  Expansion:  {new_chars/max(old_chars,1):.1f}x")
    print(f"\n  ── Old prompt features ──")
    print(f"    DDL schema     : {'✅' if 'CREATE TABLE' in old_prompt else '❌'}")
    print(f"    M-Schema       : {'✅' if '【' in old_prompt else '❌'}")
    print(f"    Table summary  : {'✅' if '•' in old_prompt else '❌'}")
    print(f"    Value hints    : {'✅' if 'Sample column' in old_prompt else '❌'}")
    print(f"    Column linking : {'✅' if '★' in old_prompt else '❌'}")
    print(f"    Value injection: {'✅' if 'Detected values' in old_prompt else '❌'}")
    print(f"    Sample rows    : {'✅' if 'Sample data' in old_prompt else '❌'}")
    print(f"    English hint   : {'✅' if '(English:' in old_prompt else '❌'}")

    print(f"\n  ── New prompt features ──")
    print(f"    DDL schema     : {'✅' if 'CREATE TABLE' in new_prompt else '❌'}")
    print(f"    M-Schema       : {'✅' if '【' in new_prompt else '❌'}")
    print(f"    Table summary  : {'✅' if '•' in new_prompt else '❌'}")
    print(f"    Value hints    : {'✅' if 'Sample column' in new_prompt else '❌'}")
    print(f"    Column linking : {'✅' if '★' in new_prompt else '❌'}")
    print(f"    Value injection: {'✅' if 'Detected values' in new_prompt else '❌'}")
    print(f"    Sample rows    : {'✅' if 'Sample data' in new_prompt else '❌'}")
    print(f"    English hint   : {'✅' if '(English:' in new_prompt else '❌'}")

print("\n✅ Training preview complete — prompts are aligned with inference format.")


## 🔄 Load Model + Apply QLoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"🔄 Loading {FINETUNE_7B_BASE}...")

try:
    tokenizer_7b = AutoTokenizer.from_pretrained(FINETUNE_7B_BASE, trust_remote_code=True)
except Exception:
    tokenizer_7b = AutoTokenizer.from_pretrained(FINETUNE_7B_BASE, trust_remote_code=True, use_fast=False)

if tokenizer_7b.pad_token is None:
    eot_id = tokenizer_7b.convert_tokens_to_ids("<|endoftext|>")
    if eot_id != tokenizer_7b.unk_token_id:
        tokenizer_7b.pad_token = "<|endoftext|>"
    else:
        tokenizer_7b.pad_token = tokenizer_7b.eos_token
tokenizer_7b.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model_7b = AutoModelForCausalLM.from_pretrained(
    FINETUNE_7B_BASE, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True,
)
model_7b = prepare_model_for_kbit_training(model_7b, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK_7B, lora_alpha=LORA_ALPHA_7B,
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model_7b = get_peft_model(model_7b, lora_config)

trainable = sum(p.numel() for p in model_7b.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_7b.parameters())
print(f"✅ Model loaded with QLoRA")
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


---
## 📦 Section F: Prepare Training Dataset (Prompt-Aligned)

Uses `TIER1_SYSTEM` for consistency. Adds token length diagnostics.

In [ ]:

# This REPLACES your existing "📦 Prepare Training Dataset" cell.
# Only change: uses TIER1_SYSTEM instead of SQL_ONLY_SYSTEM for
# consistency (they are identical, but this makes intent clear).
# ═══════════════════════════════════════════════════════════════════



#@title 📦 Prepare Training Dataset (Prompt-Aligned) { display-mode: "form" }

from datasets import Dataset

print(f"📦 Formatting {len(training_data)} training examples...")

formatted_7b = []
skipped_long = 0

for ex in training_data:
    messages = [
        {"role": "system", "content": TIER1_SYSTEM},   # ← aligned system prompt
        {"role": "user", "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    try:
        token_ids = tokenizer_7b.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=False,
        )
        if len(token_ids) <= MAX_SEQ_LENGTH_7B:
            text = tokenizer_7b.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False,
            )
            formatted_7b.append({"text": text})
        else:
            skipped_long += 1
    except Exception:
        text = (
            f"<|im_start|>system\n{TIER1_SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{ex['input']}<|im_end|>\n"
            f"<|im_start|>assistant\n{ex['output']}<|im_end|>"
        )
        if len(text) / 2.5 <= MAX_SEQ_LENGTH_7B:
            formatted_7b.append({"text": text})
        else:
            skipped_long += 1

print(f"✅ Ready: {len(formatted_7b)} examples (skipped {skipped_long} too long)")

if skipped_long > len(training_data) * 0.05:
    print(f"   ⚠️  {skipped_long} examples ({100*skipped_long/len(training_data):.1f}%) "
          f"exceeded MAX_SEQ_LENGTH={MAX_SEQ_LENGTH_7B}")
    print(f"   💡 The aligned prompts are longer. Consider increasing MAX_SEQ_LENGTH_7B.")
    print(f"      Current: {MAX_SEQ_LENGTH_7B} → Suggested: 4096 (if not already)")

full_dataset = Dataset.from_list(formatted_7b)

# Hold out 5% for overfitting detection
split = full_dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
val_ds = split["test"]
print(f"   Train: {len(train_ds)} | Val: {len(val_ds)} (5% held out for loss monitoring)")

# ── Token length distribution of actual formatted examples ──────
print(f"\n   📊 Actual token lengths (chat-formatted):")
sample_lens = []
for ex in formatted_7b[:300]:
    try:
        tids = tokenizer_7b.encode(ex["text"])
        sample_lens.append(len(tids))
    except:
        pass
if sample_lens:
    print(f"      Min: {min(sample_lens)}, Max: {max(sample_lens)}, "
          f"Avg: {sum(sample_lens)//len(sample_lens)}")
    print(f"      Median: {sorted(sample_lens)[len(sample_lens)//2]}")
    p95 = sorted(sample_lens)[int(len(sample_lens)*0.95)]
    print(f"      P95: {p95}  {'✅' if p95 <= MAX_SEQ_LENGTH_7B else '⚠️ > MAX_SEQ_LENGTH'}")


## 🎓 Train (QLoRA Fine-Tuning)

In [ ]:
from trl import SFTTrainer, SFTConfig
training_args = SFTConfig(
    output_dir=str(ADAPTER_7B_OUTPUT),
    num_train_epochs=NUM_EPOCHS_7B,
    per_device_train_batch_size=PER_DEVICE_BATCH_7B,
    gradient_accumulation_steps=GRAD_ACCUM_7B,
    learning_rate=LEARNING_RATE_7B,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True, fp16=False,
    max_grad_norm=1.0,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    max_length=MAX_SEQ_LENGTH_7B,
    dataset_text_field="text",
    report_to="none",
    seed=42,
    eval_strategy="epoch",
    per_device_eval_batch_size=PER_DEVICE_BATCH_7B,
)

trainer = SFTTrainer(
    model=model_7b, processing_class=tokenizer_7b,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=training_args,
)

print(f"🎓 Starting QLoRA fine-tuning on train.json...")
print(f"   Base: {FINETUNE_7B_BASE}")
print(f"   Prompt: {'🆕 Enhanced' if USE_ENHANCED_PROMPT else '📋 Base'}")
print(f"   Data: {len(train_ds)} train + {len(val_ds)} val")
print(f"   Epochs: {NUM_EPOCHS_7B}")
print(f"   Effective batch: {PER_DEVICE_BATCH_7B * GRAD_ACCUM_7B}")
print(f"   LoRA: rank={LORA_RANK_7B}, alpha={LORA_ALPHA_7B}")
print(f"   LR: {LEARNING_RATE_7B}")
print(f"   Max seq length: {MAX_SEQ_LENGTH_7B}")
print()

trainer.train()

# Save
model_7b.save_pretrained(str(ADAPTER_7B_OUTPUT))
tokenizer_7b.save_pretrained(str(ADAPTER_7B_OUTPUT))
print(f"\n✅ Adapter saved → {ADAPTER_7B_OUTPUT}")

# Log metrics
if trainer.state.log_history:
    tl = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    el = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    if tl: print(f"   Initial loss: {tl[0]:.4f} → Final: {tl[-1]:.4f}")
    if el:
        print(f"   Val losses: {' → '.join(f'{l:.4f}' for l in el)}")
        if len(el) > 1 and el[-1] > el[-2]:
            print(f"   ⚠️  Val loss increased — possible overfitting")


## 💾 Save Adapter & Tokenizer to Google Drive

In [ ]:
#@title 💾 Save Adapter & Tokenizer to Google Drive { display-mode: "form" }

import shutil
DRIVE_SAVE_DIR = "/content/drive/MyDrive/vietnamese_text2sql_7b_adapter_v5_aligned"  #@param {type:"string"}
ADAPTER_PATH = str(ADAPTER_7B_OUTPUT)

# Validate adapter exists locally
assert os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")), \
    f"❌ Adapter not found at {ADAPTER_PATH}"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy all adapter files
print(f"📂 Source: {ADAPTER_PATH}")
print(f"📂 Destination: {DRIVE_SAVE_DIR}\n")

count = 0
total_size = 0
for f in os.listdir(ADAPTER_PATH):
    src = os.path.join(ADAPTER_PATH, f)
    if os.path.isfile(src):
        dst = os.path.join(DRIVE_SAVE_DIR, f)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(src) / 1024 / 1024
        total_size += size_mb
        print(f"   ✅ {f} ({size_mb:.1f} MB)")
        count += 1

# Verify key files made it
for required in ["adapter_config.json", "adapter_model.safetensors"]:
    assert os.path.exists(os.path.join(DRIVE_SAVE_DIR, required)), \
        f"❌ Missing {required} in destination"

print(f"\n🎉 Saved {count} files ({total_size:.1f} MB total) to Google Drive")
print(f"   Path: {DRIVE_SAVE_DIR}")


## 🧹 Free Training Memory

In [ ]:

#@title 🧹 Free Training Memory { display-mode: "form" }

import torch

try:
  del trainer
except:
  print('Cannot delete trainer')

try:
  del model_7b
except:
  print('Cannot delete model_7b')

try:
  del train_ds
except:
  print('Cannot delete train_ds')

try:
  del val_ds
except:
  print('Cannot delete val_ds')

try:
  del full_dataset
except:
  print('Cannot delete full_dataset')


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"   GPU after cleanup: {torch.cuda.memory_allocated()/1024**3:.1f}GB")
print("✅ Training memory freed")
